# 08 · Verifica: e il certificato vecchio?
*Blocco 3 · Certificati. Passo di verifica (solo letture).*

Il device ha un certificato nuovo e uno sostituito. Cosa risponde l'Hub a ciascuno?

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

In [ ]:
$id = 'robot-02'
$dev = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '07-device-rinnovo'

## Il certificato nuovo

In [ ]:
Show-HubAcceptance -Label 'certificato rinnovato' -Hub $dev.AssignedHub -DeviceId $id -StateDir $dev.Dir

certificato rinnovato                                HTTP 204  (accettato)

## Il certificato sostituito
Lo prendiamo dalla cartella `history` del device e lo mettiamo in una cartella temporanea, con la sua chiave.

In [ ]:
$hist = Join-Path $dev.Dir 'history'
$vecchio = Join-Path $env:TEMP "vecchio-$([guid]::NewGuid().ToString('N').Substring(0, 6))"
New-Item -ItemType Directory $vecchio | Out-Null
Copy-Item (Get-ChildItem $hist -Filter '*issued-chain.pem' | Select-Object -First 1).FullName (Join-Path $vecchio 'issued-chain.pem')
Copy-Item (Get-ChildItem $hist -Filter '*device-key.pem' | Select-Object -First 1).FullName (Join-Path $vecchio 'device-key.pem')
Show-HubAcceptance -Label 'certificato sostituito dal rinnovo' -Hub $dev.AssignedHub -DeviceId $id -StateDir $vecchio
Remove-Item $vecchio -Recurse -Force

certificato sostituito dal rinnovo                   HTTP 204  (accettato)

## Cosa abbiamo visto
- L'Hub accetta il certificato nuovo **e anche quello sostituito**, fino alla scadenza: il rinnovo non invalida il precedente.
- Per escludere un certificato prima della scadenza serve la **revoca** (notebook 09).

Prossimo notebook: **09**, la revoca dei certificati.